# Notebook 1: Anthropic API Fundamentals & Tool Definitions

**Goal**: Master the building blocks before the interview starts. By the end, you'll be able to:
- Set up the Anthropic client
- Send messages and understand responses
- Define tools with proper JSON Schema
- Understand `stop_reason`, content blocks, and the overall API contract

---

## 1.1 Setup & Installation

In Colab, you'll likely get starter code with the client already set up. But you need to know this cold.

In [ ]:
# Run this cell first in Colab
!pip install anthropic pydantic -q

In [ ]:
import anthropic
import json

# Option 1: Set your API key directly (for Colab)
# client = anthropic.Anthropic(api_key="sk-ant-...")

# Option 2: From environment variable (preferred)
# export ANTHROPIC_API_KEY=sk-ant-...
client = anthropic.Anthropic()

print("Client ready.")

## 1.2 The Messages API — Core Contract

Every interaction with Claude goes through `client.messages.create()`. Here's the anatomy:

```python
response = client.messages.create(
    model="claude-sonnet-4-20250514",    # Model ID
    max_tokens=1024,                      # Max output tokens
    system="You are a helpful assistant.", # Optional system prompt
    tools=[...],                           # Optional tool definitions
    tool_choice={"type": "auto"},         # Optional: auto|any|tool|none
    messages=[                             # Required: conversation history
        {"role": "user", "content": "Hello!"}
    ]
)
```

### Key things to remember:
- Messages alternate: `user` → `assistant` → `user` → ...
- The `content` field can be a **string** OR a **list of content blocks**
- Response has: `id`, `model`, `role`, `content`, `stop_reason`, `usage`

In [ ]:
# Basic message — no tools
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=256,
    messages=[{"role": "user", "content": "What is 2 + 2? Reply in one word."}]
)

# Explore the response object
print("Type:", type(response))
print("Stop reason:", response.stop_reason)
print("Role:", response.role)
print("Content blocks:", response.content)
print("Text:", response.content[0].text)
print("Usage:", response.usage)

### Exercise 1.2a: Examine the response

Run the cell above and answer:
1. What is `response.stop_reason`? → should be `"end_turn"`
2. What type is `response.content`? → it's a **list** of content blocks
3. How do you get the text? → `response.content[0].text`
4. What's in `response.usage`? → `input_tokens` and `output_tokens`

## 1.3 Content Blocks — The Key Abstraction

Claude's messages aren't just strings. They're lists of **typed content blocks**:

| Block Type | Found In | Purpose |
|---|---|---|
| `text` | `user` or `assistant` | Regular text |
| `tool_use` | `assistant` only | Claude wants to call a tool |
| `tool_result` | `user` only | You returning tool output to Claude |
| `image` | `user` only | Image input |

**This is the most important concept**: A single assistant message can contain BOTH text AND tool_use blocks.

```python
# Example: Claude's response when it wants to use a tool
response.content = [
    {"type": "text", "text": "I'll look that up for you."},
    {"type": "tool_use", "id": "toolu_abc123", "name": "get_weather", "input": {"location": "NYC"}}
]
response.stop_reason = "tool_use"  # <-- This tells you Claude wants to use a tool
```

## 1.4 Defining Tools — JSON Schema

Tools are defined as dictionaries with three required fields:

```python
tool = {
    "name": "tool_name",           # Must match: ^[a-zA-Z0-9_-]{1,64}$
    "description": "...",           # Detailed description (CRITICAL for performance)
    "input_schema": {               # JSON Schema object
        "type": "object",
        "properties": {
            "param1": {
                "type": "string",
                "description": "What this param is for"
            }
        },
        "required": ["param1"]
    }
}
```

### JSON Schema types you'll use:
- `"type": "string"` — text values
- `"type": "number"` or `"integer"` — numeric values
- `"type": "boolean"` — true/false
- `"type": "array"`, `"items": {...}` — lists
- `"type": "object"`, `"properties": {...}` — nested objects
- `"enum": ["a", "b", "c"]` — constrained set of values

In [ ]:
# EXAMPLE: A well-defined tool
calculator_tool = {
    "name": "calculator",
    "description": (
        "Performs basic arithmetic operations on two numbers. "
        "Supports addition, subtraction, multiplication, and division. "
        "Use this tool whenever the user asks for a mathematical calculation. "
        "Returns the numeric result as a string."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "operation": {
                "type": "string",
                "enum": ["add", "subtract", "multiply", "divide"],
                "description": "The arithmetic operation to perform"
            },
            "a": {
                "type": "number",
                "description": "The first operand"
            },
            "b": {
                "type": "number",
                "description": "The second operand"
            }
        },
        "required": ["operation", "a", "b"]
    }
}

print(json.dumps(calculator_tool, indent=2))

### Exercise 1.4a: Write a tool definition

Define a tool called `search_database` that:
- Takes a `query` (required string) — the search query
- Takes a `max_results` (optional integer, default 10) — how many results to return
- Takes a `category` (optional, one of: "users", "products", "orders") — filter by category

Write a detailed description. Try it yourself before looking at the solution below.

In [ ]:
# YOUR ANSWER HERE
search_tool = {
    "name": "search_database",
    "description": "",  # Fill this in
    "input_schema": {
        "type": "object",
        "properties": {
            # Fill in the properties
        },
        "required": []  # Fill in required fields
    }
}

In [ ]:
# SOLUTION
search_tool = {
    "name": "search_database",
    "description": (
        "Searches the application database for records matching a query string. "
        "Returns a list of matching records with their IDs and key fields. "
        "Use this when the user asks to find, look up, or search for specific data. "
        "Results are returned in relevance order. "
        "If no category is specified, searches across all record types."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query string to match against records"
            },
            "max_results": {
                "type": "integer",
                "description": "Maximum number of results to return. Defaults to 10 if not specified."
            },
            "category": {
                "type": "string",
                "enum": ["users", "products", "orders"],
                "description": "Filter results to a specific category. If omitted, searches all categories."
            }
        },
        "required": ["query"]
    }
}

print(json.dumps(search_tool, indent=2))

## 1.4b SPEED HACK: Pydantic Tool Helper

Writing JSON schemas by hand is slow and error-prone. Pydantic models produce **identical JSON Schema output** — and they're much faster to type.

### The helper (5 lines — memorize this):

```python
from pydantic import BaseModel, Field
from typing import Literal
import re

def tool(model: type[BaseModel]):
    name = re.sub(r'(?<!^)(?=[A-Z])', '_', model.__name__).lower()
    return {"name": name, "description": model.__doc__ or "",
            "input_schema": model.model_json_schema()}
```

### How it maps:
| Pydantic | JSON Schema |
|----------|-------------|
| `str` | `{"type": "string"}` |
| `int` | `{"type": "integer"}` |
| `float` | `{"type": "number"}` |
| `bool` | `{"type": "boolean"}` |
| `Literal["a","b"]` | `{"type": "string", "enum": ["a","b"]}` |
| `list[str]` | `{"type": "array", "items": {"type": "string"}}` |
| `Field(description="...")` | `{"description": "..."}` |
| No default → in `required` | Default value → optional |
| Docstring | Tool `description` |
| CamelCase class name | `snake_case` tool name |

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
import re

# ===== THE HELPER — 5 lines, memorize this =====
def tool(model: type[BaseModel]):
    """Convert Pydantic model → Anthropic tool definition."""
    name = re.sub(r'(?<!^)(?=[A-Z])', '_', model.__name__).lower()
    return {
        "name": name,
        "description": model.__doc__ or "",
        "input_schema": model.model_json_schema()
    }

# ===== DEFINE TOOLS IN SECONDS =====

class Calculator(BaseModel):
    """Perform basic arithmetic on two numbers. Returns the numeric result."""
    operation: Literal["add", "subtract", "multiply", "divide"] = Field(description="Math operation")
    a: float = Field(description="First number")
    b: float = Field(description="Second number")

class SearchDatabase(BaseModel):
    """Search records by keyword. Returns matching results in relevance order."""
    query: str = Field(description="Search keyword")
    max_results: int = Field(default=10, description="Max items to return")
    category: Literal["users", "products", "orders"] | None = Field(default=None, description="Filter by type")

class LookupUser(BaseModel):
    """Look up a user by ID. Returns profile info including name and email."""
    user_id: int = Field(description="Unique user identifier")

# Convert to tool definitions — done!
pydantic_tools = [tool(Calculator), tool(SearchDatabase), tool(LookupUser)]

# Compare: Pydantic vs hand-written
print("=== Pydantic-generated tool definition ===")
print(json.dumps(tool(Calculator), indent=2))
print(f"\n3 tools defined in ~15 lines vs ~60 lines of raw JSON schema")

## 1.5 Making a Tool Use Request

When you pass `tools` to `client.messages.create()`, Claude can choose to use them. Let's see it in action.

In [ ]:
# Send a request with a tool
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[calculator_tool],
    messages=[{"role": "user", "content": "What is 1234 * 5678?"}]
)

print("Stop reason:", response.stop_reason)
print()

for block in response.content:
    if block.type == "text":
        print(f"[TEXT]: {block.text}")
    elif block.type == "tool_use":
        print(f"[TOOL_USE]:")
        print(f"  id:    {block.id}")
        print(f"  name:  {block.name}")
        print(f"  input: {block.input}")

### Key observations:

1. **`stop_reason` is `"tool_use"`** — This means Claude stopped because it wants to call a tool
2. **Content has both `text` AND `tool_use` blocks** — Claude explains what it's doing, then requests the tool
3. **`tool_use` block has `id`, `name`, `input`** — You need ALL of these to process the call
4. **The `id` is critical** — You must match it when returning `tool_result`

## 1.6 Returning Tool Results

After executing the tool, you send the result back in a `user` message with a `tool_result` block.

### The message structure:
```python
{
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": "toolu_abc123",  # MUST match the tool_use id
            "content": "The result string"   # Your tool's output
        }
    ]
}
```

### Rules:
- `tool_result` must come in the VERY NEXT `user` message after the `assistant` tool_use
- `tool_use_id` MUST match the `id` from the tool_use block
- `tool_result` blocks must come FIRST in the content array (before any text)
- For errors, add `"is_error": true`

In [ ]:
# Complete round-trip: request → tool_use → tool_result → final answer

# Step 1: Initial request
messages = [{"role": "user", "content": "What is 1234 * 5678?"}]

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[calculator_tool],
    messages=messages
)

print("Step 1 - Claude wants tool:", response.stop_reason)

# Step 2: Extract tool call and execute it
tool_use_block = next(b for b in response.content if b.type == "tool_use")
print(f"Tool: {tool_use_block.name}, Input: {tool_use_block.input}")

# Actually run the tool
inp = tool_use_block.input
if inp["operation"] == "multiply":
    result = inp["a"] * inp["b"]
elif inp["operation"] == "add":
    result = inp["a"] + inp["b"]
elif inp["operation"] == "subtract":
    result = inp["a"] - inp["b"]
elif inp["operation"] == "divide":
    result = inp["a"] / inp["b"]

print(f"Tool result: {result}")

# Step 3: Send result back to Claude
messages.append({"role": "assistant", "content": response.content})  # Add Claude's tool_use message
messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": tool_use_block.id,
            "content": str(result)
        }
    ]
})

# Step 4: Get Claude's final answer
final_response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[calculator_tool],
    messages=messages
)

print(f"\nFinal stop reason: {final_response.stop_reason}")
print(f"Final answer: {final_response.content[0].text}")

## 1.7 `stop_reason` Cheat Sheet

| `stop_reason` | What it means | What to do |
|---|---|---|
| `"end_turn"` | Claude is done talking | Return the response to the user |
| `"tool_use"` | Claude wants to call a tool | Extract tool call, execute it, return `tool_result` |
| `"max_tokens"` | Hit the token limit | Retry with higher `max_tokens` |
| `"stop_sequence"` | Hit a custom stop sequence | Handle accordingly |
| `"pause_turn"` | Server tool paused (web search) | Continue conversation with content |

**In your interview, you'll mainly deal with `"end_turn"` and `"tool_use"`.**

## 1.8 `tool_choice` — Controlling Tool Usage

You can control whether/how Claude uses tools:

```python
# Let Claude decide (default when tools are provided)
tool_choice={"type": "auto"}

# Force Claude to use at least one tool
tool_choice={"type": "any"}

# Force a specific tool
tool_choice={"type": "tool", "name": "calculator"}

# Prevent tool use (default when no tools provided)
tool_choice={"type": "none"}
```

In the interview, you'll likely use `auto` (the default) — just don't pass `tool_choice` at all.

In [ ]:
# Force Claude to use the calculator, even for a simple question
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[calculator_tool],
    tool_choice={"type": "tool", "name": "calculator"},
    messages=[{"role": "user", "content": "What is 2 + 2?"}]
)

print("Stop reason:", response.stop_reason)
for block in response.content:
    print(f"  {block.type}: {block}")

## 1.9 Common Mistakes to Avoid

### Mistake 1: Forgetting to include tools in follow-up calls
```python
# BAD: Tools missing from second call
response2 = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    # tools=...  <-- MISSING! Must include tools in every call
    messages=messages
)
```

### Mistake 2: Wrong tool_use_id
```python
# BAD: ID doesn't match
{"type": "tool_result", "tool_use_id": "wrong_id", "content": "..."}
```

### Mistake 3: Text before tool_result in content array
```python
# BAD: text before tool_result
{"role": "user", "content": [
    {"type": "text", "text": "Here are results:"},  # <-- NO!
    {"type": "tool_result", "tool_use_id": "...", "content": "..."}
]}

# GOOD: tool_result first
{"role": "user", "content": [
    {"type": "tool_result", "tool_use_id": "...", "content": "..."},
    {"type": "text", "text": "Any additional context"}  # <-- OK after
]}
```

### Mistake 4: Sending tool result as a string instead of content block
```python
# BAD
{"role": "user", "content": "The result is 42"}

# GOOD
{"role": "user", "content": [{"type": "tool_result", "tool_use_id": "...", "content": "42"}]}
```

## 1.10 Quick Reference Card

Print this or keep it open during the interview:

```python
# === TOOL DEFINITION TEMPLATE ===
tool = {
    "name": "my_tool",
    "description": "Detailed description of what this tool does...",
    "input_schema": {
        "type": "object",
        "properties": {
            "param": {"type": "string", "description": "..."}
        },
        "required": ["param"]
    }
}

# === API CALL ===
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[tool],
    messages=[{"role": "user", "content": "..."}]
)

# === CHECK IF TOOL USE ===
if response.stop_reason == "tool_use":
    tool_block = next(b for b in response.content if b.type == "tool_use")
    # tool_block.id, tool_block.name, tool_block.input

# === RETURN RESULT ===
tool_result_message = {
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": tool_block.id,
        "content": str(result)
    }]
}
```

### Exercise 1.10a: Full Round-Trip from Scratch

Without looking at the code above, write a complete round-trip:
1. Define a tool called `lookup_user` that takes a `user_id` (required integer) and returns user info
2. Send a message asking "Who is user 42?"
3. Handle the tool_use response
4. Return a fake result: `{"name": "Alice", "email": "alice@example.com"}`
5. Print Claude's final response

In [ ]:
# YOUR ANSWER: Write the full round-trip here
# ...


In [ ]:
# SOLUTION

# 1. Define the tool
lookup_user_tool = {
    "name": "lookup_user",
    "description": (
        "Looks up a user in the database by their unique user ID. "
        "Returns the user's profile information including name, email, and account status. "
        "Use this when the user asks about a specific user by ID."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "user_id": {
                "type": "integer",
                "description": "The unique identifier of the user to look up"
            }
        },
        "required": ["user_id"]
    }
}

# 2. Send initial request
messages = [{"role": "user", "content": "Who is user 42?"}]

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[lookup_user_tool],
    messages=messages
)

# 3. Handle tool_use
assert response.stop_reason == "tool_use"
tool_block = next(b for b in response.content if b.type == "tool_use")
print(f"Claude wants to call {tool_block.name} with {tool_block.input}")

# 4. Return fake result
fake_result = json.dumps({"name": "Alice", "email": "alice@example.com"})

messages.append({"role": "assistant", "content": response.content})
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": tool_block.id,
        "content": fake_result
    }]
})

# 5. Get final response
final = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[lookup_user_tool],
    messages=messages
)

print(f"\nClaude's answer: {final.content[0].text}")

---

## Summary — What You Must Know Cold

1. **`client.messages.create()`** — model, max_tokens, tools, messages
2. **Tool definition** — name, description (detailed!), input_schema (JSON Schema)
3. **Response content is a list of blocks** — `text` and `tool_use`
4. **`stop_reason == "tool_use"`** means Claude wants you to execute a tool
5. **`tool_result` block** — must include matching `tool_use_id`, goes in next `user` message
6. **Tools must be included in every API call** in the conversation

**Next: Notebook 2 — Tool Use Mechanics (handling responses, dispatching tools, error handling)**